In [1]:
import tokenizers
import torch
import torch.nn as nn
import torchmetrics
import transformers
from torch.utils.data import DataLoader

from datasets import load_dataset

In [2]:
device = "cuda"

In [3]:
torch.cuda.is_available()

True

In [4]:
imdb_dataset = load_dataset("stanfordnlp/imdb")
split = imdb_dataset["train"].train_test_split(train_size=0.8, seed=42)
imdb_train_set, imdb_valid_set = split["train"], split["test"]
imdb_test_set = imdb_dataset["test"]

In [5]:
imdb_train_set[1]["text"]

"'The Rookie' was a wonderful movie about the second chances life holds for us and also puts an emotional thought over the audience, making them realize that your dreams can come true. If you loved 'Remember the Titans', 'The Rookie' is the movie for you!! It's the feel good movie of the year and it is the perfect movie for all ages. 'The Rookie' hits a major home run!"

In [6]:
imdb_train_set[1]["label"]

1

In [7]:
print(imdb_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [8]:
print(imdb_train_set[16]["text"])

Lillian Hellman's play, adapted by Dashiell Hammett with help from Hellman, becomes a curious project to come out of gritty Warner Bros. Paul Lukas, reprising his Broadway role and winning the Best Actor Oscar, plays an anti-Nazi German underground leader fighting the Fascists, dragging his American wife and three children all over Europe before finding refuge in the States (via the Mexico border). They settle in Washington with the wife's wealthy mother and brother, though a boarder residing in the manor is immediately suspicious of the newcomers and spends an awful lot of time down at the German Embassy playing poker. It seems to take forever for this drama to find its focus, and when we realize what the heart of the material is (the wise, honest, direct refugees teaching the clueless, head-in-the-sand Americans how the world has suddenly changed), it seems a little patronizing--the viewer is quite literally put in the relatives' place, being lectured to. Lukas has several speeches i

In [9]:
bpe_model = tokenizers.models.BPE(unk_token="<unk>")
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()

special_tokens = ["<pad>", "<unk>"]
bpe_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=1000, special_tokens=special_tokens
)
train_reviews = [review["text"].lower() for review in imdb_train_set]
bpe_tokenizer.train_from_iterator(train_reviews, bpe_trainer)

In [10]:
print(len(train_reviews))

20000


In [11]:
some_review = "what an awesome movie! 😀"

bpe_encoding = bpe_tokenizer.encode(some_review)
bpe_encoding

Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [12]:
bpe_encoding.tokens

['what', 'an', 'aw', 'es', 'ome', 'movie', '!', '<unk>']

In [13]:
bpe_token_ids = bpe_encoding.ids

In [14]:
bpe_token_ids

[303, 139, 373, 149, 240, 211, 4, 1]

In [15]:
bpe_tokenizer.decode(bpe_token_ids)

'what an aw es ome movie !'

In [16]:
bpe_encoding.offsets

[(0, 4), (5, 7), (8, 10), (10, 12), (12, 15), (16, 21), (21, 22), (23, 24)]

In [17]:
bpe_tokenizer.encode_batch(train_reviews[:3])

[Encoding(num_tokens=281, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=114, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=285, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]

In [18]:
bpe_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
bpe_tokenizer.enable_truncation(max_length=500)

In [19]:
bpe_encodings = bpe_tokenizer.encode_batch(train_reviews[:3])
bpe_batch_ids = torch.tensor([encoding.ids for encoding in bpe_encodings])
bpe_batch_ids

tensor([[159, 402, 176, 246,  61, 782, 156, 737, 252,  42, 239,  51, 154, 460,
         917,  17, 272, 156, 737, 576, 215, 976, 275,  42, 199,  44, 554,  42,
         192, 585,  57, 160, 259, 170, 157, 143, 138, 159, 402,  11, 589, 152,
           5, 819, 168, 230,   5, 521, 924, 981, 962, 250,  61,  10,  60, 426,
         526, 959,  60, 138, 199, 150, 319,  15, 363, 141, 957, 694,  47, 696,
          61, 875, 138, 960, 337, 414, 140, 157, 385, 174, 433, 161, 221, 145,
         213,  17, 549,  15, 151,  10,  60,  55, 416, 146, 407, 144, 182, 303,
         151, 141,  17, 138, 547, 538, 528, 768,  54, 335,  42, 203,  44, 270,
          46, 153, 876, 141, 919, 233, 522, 172, 141, 719, 162, 807, 279,  17,
         138,  45,  66,  55, 188, 989, 156, 378, 698, 301, 296, 689, 212, 558,
         926, 148,  17,  44, 270,  46, 141,  47, 279, 302, 171, 152, 787,  15,
         153, 522, 172, 766, 205, 156, 234, 677, 161, 139, 513, 146, 370, 251,
         219, 162, 197, 162, 166,  50, 265,  47, 266

In [20]:
attention_mask = torch.tensor([encoding.attention_mask for encoding in bpe_encodings])

In [21]:
attention_mask

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [22]:
lengths = attention_mask.sum(dim=-1)
lengths

tensor([281, 114, 285])

In [23]:
# BPPE Tokenization
tokenizers.pre_tokenizers.ByteLevel().pre_tokenize_str(some_review)

[('Ġwhat', (0, 4)),
 ('Ġan', (4, 7)),
 ('Ġawesome', (7, 15)),
 ('Ġmovie', (15, 21)),
 ('!', (21, 22)),
 ('ĠðŁĺĢ', (22, 24))]

In [24]:
bbpe_model = tokenizers.models.BPE(unk_token="<unk>")
bbpe_tokenzier = tokenizers.Tokenizer(bbpe_model)
bbpe_tokenzier.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()
bbpe_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=1000, special_tokens=special_tokens
)
bbpe_tokenzier.train_from_iterator(train_reviews, bbpe_trainer)

In [25]:
bbpe_encoding = bbpe_tokenzier.encode(some_review)
bbpe_tokens = bbpe_encoding.tokens
bbpe_tokens

['Ġwhat',
 'Ġan',
 'Ġaw',
 'es',
 'ome',
 'Ġmovie',
 '!',
 'Ġ',
 '<unk>',
 'Ł',
 'ĺ',
 'Ģ']

In [26]:
bbpe_token_ids = bbpe_encoding.ids
bbpe_token_ids

[354, 216, 561, 148, 244, 232, 2, 107, 1, 125, 119, 108]

In [27]:
bbpe_decoded = bbpe_tokenzier.decode(bbpe_token_ids)
bbpe_decoded

'Ġwhat Ġan Ġaw es ome Ġmovie ! Ġ Ł ĺ Ģ'

In [28]:
bbpe_decoded.replace(" ", "").replace("Ġ", " ").strip()

'what an awesome movie! ŁĺĢ'

In [29]:
wp_model = tokenizers.models.WordPiece(unk_token="<unk>")
wp_tokenizer = tokenizers.Tokenizer(wp_model)
wp_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
wp_trainer = tokenizers.trainers.WordPieceTrainer(
    vocab_size=1000, special_tokens=special_tokens
)
wp_tokenizer.train_from_iterator(train_reviews, wp_trainer)

In [30]:
wp_encoding = wp_tokenizer.encode(some_review)
wp_tokens = wp_encoding.tokens
wp_tokens

['what', 'an', 'aw', '##es', '##ome', 'movie', '!', '<unk>']

In [31]:
wp_token_ids = wp_encoding.ids
wp_token_ids

[443, 312, 635, 257, 354, 331, 4, 1]

In [32]:
wp_decoded = wp_tokenizer.decode(wp_token_ids)
wp_decoded

'what an aw ##es ##ome movie !'

In [33]:
wp_decoded.replace(" ##", "")

'what an awesome movie !'

In [34]:
unigram_model = tokenizers.models.Unigram()
unigram_tokenizer = tokenizers.Tokenizer(unigram_model)
unigram_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
unigram_trainer = tokenizers.trainers.UnigramTrainer(
    vocab_size=1000, special_tokens=special_tokens, unk_token="<unk>"
)
unigram_tokenizer.train_from_iterator(train_reviews, unigram_trainer)

In [35]:
unigram_encoding = unigram_tokenizer.encode(some_review)
unigram_tokens = unigram_encoding.tokens
unigram_tokens


['what', 'an', 'a', 'w', 'e', 'some', 'movie', '!', '😀']

In [36]:
unigram_token_ids = unigram_encoding.ids[:10]
unigram_token_ids

[79, 37, 4, 40, 6, 70, 46, 74, 1]

In [37]:
unigram_tokenizer.decode(unigram_token_ids)

'what an a w e some movie !'

In [38]:
# Reusing Pretrained Tokenizers
gpt2_tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")
gpt2_encoding = gpt2_tokenizer(train_reviews[:3], truncation=True, max_length=500)

In [39]:
gpt2_token_ids = gpt2_encoding["input_ids"][0][:10]
gpt2_token_ids

[14247, 35030, 1690, 423, 257, 1688, 8046, 13, 484, 1690]

In [40]:
gpt2_tokenizer.decode(gpt2_token_ids)

'stage adaptations often have a major fault. they often'

In [41]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
bert_encoding = bert_tokenizer(
    train_reviews[:3],
    padding=True,
    truncation=True,
    max_length=500,
    return_tensors="pt",
)

In [42]:
bert_encoding["input_ids"]

tensor([[  101,  2754, 17241,  2411,  2031,  1037,  2350,  6346,  1012,  2027,
          2411,  2272,  2041,  2559,  2066,  1037,  2143,  4950,  2001,  3432,
          2872,  2006,  1996,  2754,  1006,  2107,  2004,  1000,  2305,  2388,
          1000,  1007,  1012, 11430, 11320, 11368,  1005,  1055,  3257,  7906,
          1996,  2143,  4142,  1010,  2029,  2003,  2926,  3697,  2144,  1996,
          3861,  3253,  2032,  2053,  2613,  4119,  1012,  2145,  1010,  2009,
          1005,  1055,  3835,  2000,  2298,  2012,  2005,  2054,  2009,  2003,
          1012,  1996,  6370,  2090,  2745, 19881,  1998,  5696, 20726,  2003,
          3243,  8235,  1012,  1996, 10949,  1997,  2037,  3276,  2024, 11341,
          1012, 19881,  2003, 10392,  2004,  2467,  1010,  1998, 20726,  4152,
          2028,  1997,  2010,  2261,  9592,  2000,  2428,  2552,  1012,  1026,
          7987,  1013,  1028,  1026,  7987,  1013,  1028,  1045, 18766,  2008,
          1045,  1005,  2310,  2196,  2464, 11209, 2

In [43]:
bert_encoding["attention_mask"]

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [44]:
albert_tokenizer = transformers.AutoTokenizer.from_pretrained("albert-base-v2")
albert_encoding = albert_tokenizer(
    train_reviews[:3],
    padding=True,
    truncation=True,
    max_length=500,
    add_special_tokens=False,
    return_tensors="pt",
)
albert_token_ids = albert_encoding["input_ids"]
albert_token_ids

tensor([[  876,  5004,    18,   478,    57,    21,   394,  4173,     9,    59,
           478,   340,    70,   699,   101,    21,   171,  3336,    23,  1659,
          1037,    27,    14,   876,    13,     5,  4289,    28,    13,     7,
          4893,   449,     7,     6,     9, 12508,  1612,  5909,    22,    18,
          1400,  8968,    14,   171,  2481,    15,    56,    25,  1118,  1956,
           179,    14,  2151,  1434,    61,    90,   683,  2404,     9,   174,
            15,    32,    22,    18,  2210,    20,   361,    35,    26,    98,
            32,    25,     9,    14,  5427,   128,   832, 22427,    17,  4479,
         24604,    25,  1450,  7472,     9,    14, 12289,    16,    66,  1429,
            50, 12891,     9, 22427,    25, 10356,    28,   550,    15,    17,
         24604,  3049,    53,    16,    33,   310, 11285,    20,   510,   601,
             9,     1,  5145,    13,   118,     1,  5145,    13,   118,     1,
            49, 14586,    30,    31,    22,   195,  

In [45]:
hf_tokenizer = transformers.PreTrainedTokenizerFast(tokenizer_object=bpe_tokenizer)
hf_encodings = hf_tokenizer(
    train_reviews[:3],
    padding=True,
    truncation=True,
    max_length=500,
    return_tensors="pt",
)
hf_encodings["input_ids"]

tensor([[159, 402, 176, 246,  61, 782, 156, 737, 252,  42, 239,  51, 154, 460,
         917,  17, 272, 156, 737, 576, 215, 976, 275,  42, 199,  44, 554,  42,
         192, 585,  57, 160, 259, 170, 157, 143, 138, 159, 402,  11, 589, 152,
           5, 819, 168, 230,   5, 521, 924, 981, 962, 250,  61,  10,  60, 426,
         526, 959,  60, 138, 199, 150, 319,  15, 363, 141, 957, 694,  47, 696,
          61, 875, 138, 960, 337, 414, 140, 157, 385, 174, 433, 161, 221, 145,
         213,  17, 549,  15, 151,  10,  60,  55, 416, 146, 407, 144, 182, 303,
         151, 141,  17, 138, 547, 538, 528, 768,  54, 335,  42, 203,  44, 270,
          46, 153, 876, 141, 919, 233, 522, 172, 141, 719, 162, 807, 279,  17,
         138,  45,  66,  55, 188, 989, 156, 378, 698, 301, 296, 689, 212, 558,
         926, 148,  17,  44, 270,  46, 141,  47, 279, 302, 171, 152, 787,  15,
         153, 522, 172, 766, 205, 156, 234, 677, 161, 139, 513, 146, 370, 251,
         219, 162, 197, 162, 166,  50, 265,  47, 266

In [46]:
def collate_fn(batch, tokenizer=bert_tokenizer):
    reviews = [review["text"] for review in batch]
    labels = [[review["label"]] for review in batch]
    encodings = tokenizer(
        reviews, padding=True, truncation=True, max_length=200, return_tensors="pt"
    )
    labels = torch.tensor(labels, dtype=torch.float32)
    return encodings, labels


batch_size = 256
imdb_train_loader = DataLoader(
    imdb_train_set, batch_size=batch_size, collate_fn=collate_fn, shuffle=True
)
imdb_valid_loader = DataLoader(
    imdb_valid_set, batch_size=batch_size, collate_fn=collate_fn
)
imdb_test_loader = DataLoader(
    imdb_test_set, batch_size=batch_size, collate_fn=collate_fn
)

In [47]:
class SentimentAnalysisModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_layers=2,
        embed_dim=128,
        hidden_dim=64,
        pad_id=0,
        dropout=0.2,
    ):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        _outputs, hidden_states = self.gru(embeddings)
        return self.output(hidden_states[-1])

In [48]:
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()


def train(
    model,
    optimizer,
    loss_fn,
    metric,
    train_loader,
    valid_loader,
    n_epochs,
    patience=2,
    factor=0.5,
    epoch_callback=None,
):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor
    )
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss / (index + 1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(
            f"\rEpoch {epoch + 1}/{n_epochs},                      "
            f"train loss: {history['train_losses'][-1]:.4f}, "
            f"train metric: {history['train_metrics'][-1]:.2%}, "
            f"valid metric: {history['valid_metrics'][-1]:.2%}"
        )
    return history

In [49]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

sequences = torch.tensor([[1, 2, 0, 0], [5, 6, 7, 8]])
packed = pack_padded_sequence(
    sequences, lengths=(2, 4), enforce_sorted=False, batch_first=True
)

In [50]:
packed

PackedSequence(data=tensor([5, 1, 6, 2, 7, 8]), batch_sizes=tensor([2, 2, 1, 1]), sorted_indices=tensor([1, 0]), unsorted_indices=tensor([1, 0]))

In [51]:
padded, lengths = pad_packed_sequence(packed, batch_first=True)
padded, lengths

(tensor([[1, 2, 0, 0],
         [5, 6, 7, 8]]),
 tensor([2, 4]))

In [52]:
class SentimentAnalysisModelPackedSeq(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_layers=2,
        embed_dim=128,
        hidden_dim=64,
        pad_id=0,
        dropout=0.2,
    ):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(
            embeddings,
            lengths=lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _outputs, hidden_states = self.gru(packed)
        return self.output(hidden_states[-1])

In [53]:
torch.manual_seed(42)

vocab_size = bert_tokenizer.vocab_size
imdb_model_ps = SentimentAnalysisModelPackedSeq(vocab_size=vocab_size).to(device)

n_epochs = 10
x_entropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_ps.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

histroy = train(
    imdb_model_ps,
    optimizer,
    x_entropy,
    accuracy,
    imdb_train_loader,
    imdb_valid_loader,
    n_epochs,
)

Epoch 1/10,                      train loss: 0.6741, train metric: 57.45%, valid metric: 62.12%
Epoch 2/10,                      train loss: 0.5780, train metric: 70.30%, valid metric: 69.92%
Epoch 3/10,                      train loss: 0.4531, train metric: 79.33%, valid metric: 79.88%
Epoch 4/10,                      train loss: 0.3453, train metric: 85.69%, valid metric: 83.92%
Epoch 5/10,                      train loss: 0.2753, train metric: 89.11%, valid metric: 83.64%
Epoch 6/10,                      train loss: 0.2054, train metric: 92.32%, valid metric: 85.46%
Epoch 7/10,                      train loss: 0.1564, train metric: 94.63%, valid metric: 82.70%
Epoch 8/10,                      train loss: 0.1047, train metric: 96.58%, valid metric: 82.68%
Epoch 9/10,                      train loss: 0.0669, train metric: 98.04%, valid metric: 83.72%
Epoch 10/10,                      train loss: 0.0379, train metric: 99.20%, valid metric: 81.44%


In [54]:
class SentimentAnalysisModelPackedSeqBiDirectional(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_layers=2,
        embed_dim=128,
        hidden_dim=64,
        pad_id=0,
        dropout=0.2,
    ):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True,
        )
        self.output = nn.Linear(2 * hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(
            embeddings,
            lengths=lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _outputs, hidden_states = self.gru(packed)
        n_dims = self.output.in_features
        top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)
        return self.output(top_states)

In [55]:
torch.manual_seed(42)

vocab_size = bert_tokenizer.vocab_size
imdb_model_ps = SentimentAnalysisModelPackedSeqBiDirectional(vocab_size=vocab_size).to(
    device
)

n_epochs = 10
x_entropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_ps.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

histroy = train(
    imdb_model_ps,
    optimizer,
    x_entropy,
    accuracy,
    imdb_train_loader,
    imdb_valid_loader,
    n_epochs,
)

Epoch 1/10,                      train loss: 0.6525, train metric: 60.51%, valid metric: 70.74%
Epoch 2/10,                      train loss: 0.5117, train metric: 74.94%, valid metric: 70.90%
Epoch 3/10,                      train loss: 0.3809, train metric: 83.32%, valid metric: 71.22%
Epoch 4/10,                      train loss: 0.2806, train metric: 88.49%, valid metric: 80.44%
Epoch 5/10,                      train loss: 0.1982, train metric: 92.40%, valid metric: 80.12%
Epoch 6/10,                      train loss: 0.1416, train metric: 94.87%, valid metric: 83.66%
Epoch 7/10,                      train loss: 0.0701, train metric: 97.87%, valid metric: 81.88%
Epoch 8/10,                      train loss: 0.0359, train metric: 99.08%, valid metric: 83.12%
Epoch 9/10,                      train loss: 0.0151, train metric: 99.76%, valid metric: 83.20%
Epoch 10/10,                      train loss: 0.0075, train metric: 99.92%, valid metric: 83.60%


In [56]:
bert_model = transformers.AutoModel.from_pretrained("bert-base-uncased")
bert_model.embeddings.word_embeddings

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding(30522, 768, padding_idx=0)

In [57]:
class SentimentAnalysisModelPreEmbeds(nn.Module):
    def __init__(self, pretrained_embeddings, n_layers=2, hidden_dim=64, dropout=0.2):
        super().__init__()
        weights = pretrained_embeddings.weight.data
        self.embed = nn.Embedding.from_pretrained(weights, freeze=True)
        embed_dim = weights.shape[-1]
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True,
        )
        self.output = nn.Linear(2 * hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(
            embeddings, lengths=lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _outputs, hidden_states = self.gru(packed)
        n_dims = self.output.in_features
        top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)
        return self.output(top_states)

In [58]:
torch.manual_seed(42)
imdb_model_bert_embeds = SentimentAnalysisModelPreEmbeds(
    bert_model.embeddings.word_embeddings
).to(device)

n_epochs = 10
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bert_embeds.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train(
    imdb_model_bert_embeds,
    optimizer,
    xentropy,
    accuracy,
    imdb_train_loader,
    imdb_valid_loader,
    n_epochs,
)


Epoch 1/10,                      train loss: 0.6851, train metric: 55.18%, valid metric: 54.38%
Epoch 2/10,                      train loss: 0.6467, train metric: 64.45%, valid metric: 69.78%
Epoch 3/10,                      train loss: 0.5517, train metric: 72.78%, valid metric: 73.26%
Epoch 4/10,                      train loss: 0.4805, train metric: 76.84%, valid metric: 76.08%
Epoch 5/10,                      train loss: 0.4142, train metric: 80.99%, valid metric: 81.32%
Epoch 6/10,                      train loss: 0.3711, train metric: 83.07%, valid metric: 84.04%
Epoch 7/10,                      train loss: 0.3305, train metric: 85.32%, valid metric: 81.98%
Epoch 8/10,                      train loss: 0.3084, train metric: 86.60%, valid metric: 76.76%
Epoch 9/10,                      train loss: 0.2888, train metric: 87.63%, valid metric: 66.78%
Epoch 10/10,                      train loss: 0.2432, train metric: 89.89%, valid metric: 82.32%


In [59]:
bert_encoding = bert_tokenizer(
    train_reviews[:3],
    padding=True,
    max_length=200,
    truncation=True,
    return_tensors="pt",
)
print(bert_encoding)
bert_output = bert_model(**bert_encoding)
bert_output.last_hidden_state.shape

{'input_ids': tensor([[  101,  2754, 17241,  2411,  2031,  1037,  2350,  6346,  1012,  2027,
          2411,  2272,  2041,  2559,  2066,  1037,  2143,  4950,  2001,  3432,
          2872,  2006,  1996,  2754,  1006,  2107,  2004,  1000,  2305,  2388,
          1000,  1007,  1012, 11430, 11320, 11368,  1005,  1055,  3257,  7906,
          1996,  2143,  4142,  1010,  2029,  2003,  2926,  3697,  2144,  1996,
          3861,  3253,  2032,  2053,  2613,  4119,  1012,  2145,  1010,  2009,
          1005,  1055,  3835,  2000,  2298,  2012,  2005,  2054,  2009,  2003,
          1012,  1996,  6370,  2090,  2745, 19881,  1998,  5696, 20726,  2003,
          3243,  8235,  1012,  1996, 10949,  1997,  2037,  3276,  2024, 11341,
          1012, 19881,  2003, 10392,  2004,  2467,  1010,  1998, 20726,  4152,
          2028,  1997,  2010,  2261,  9592,  2000,  2428,  2552,  1012,  1026,
          7987,  1013,  1028,  1026,  7987,  1013,  1028,  1045, 18766,  2008,
          1045,  1005,  2310,  2196,  

torch.Size([3, 200, 768])

In [60]:
class SentimentAnalysisModelBert(nn.Module):
    def __init__(self, n_layers=2, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        embed_dim = self.bert.config.hidden_size
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        contextualized_embeddings = self.bert(**encodings).last_hidden_state
        lenghts = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(
            contextualized_embeddings,
            lengths=lenghts.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _outputs, hidden_states = self.gru(packed)
        return self.output(hidden_states[-1])


In [61]:
torch.manual_seed(42)

imdb_model_bert = SentimentAnalysisModelBert().to(device)
imdb_model_bert.bert.requires_grad_(False)

n_epochs = 4
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bert.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train(
    imdb_model_bert,
    optimizer,
    xentropy,
    accuracy,
    imdb_train_loader,
    imdb_valid_loader,
    n_epochs,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/4,                      train loss: 0.4819, train metric: 75.92%, valid metric: 87.36%
Epoch 2/4,                      train loss: 0.3005, train metric: 87.02%, valid metric: 88.50%
Epoch 3/4,                      train loss: 0.2706, train metric: 88.78%, valid metric: 88.38%
Epoch 4/4,                      train loss: 0.2510, train metric: 89.54%, valid metric: 85.18%


In [62]:
class SentimentAnalysisModelBert2(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.output = nn.Linear(self.bert.config.hidden_size,1)

    def forward(self, encodings):
        bert_output = self.bert(**encodings)
        return self.output(bert_output.last_hidden_state[:,0])

In [63]:
class SentimentAnalysisModelBert3(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.output = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, encodings):
        bert_output = self.bert(**encodings)
        return self.output(bert_output.pooler_output)

In [64]:
torch.manual_seed(42)

imdb_model_bert3 = SentimentAnalysisModelBert3().to(device)
imdb_model_bert3.bert.requires_grad_(False)

n_epochs = 5
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bert3.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train(imdb_model_bert3, optimizer, xentropy, accuracy,
                imdb_train_loader, imdb_valid_loader, n_epochs)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/5,                      train loss: 0.6590, train metric: 61.18%, valid metric: 70.94%
Epoch 2/5,                      train loss: 0.6048, train metric: 69.19%, valid metric: 60.26%
Epoch 3/5,                      train loss: 0.5702, train metric: 72.56%, valid metric: 74.18%
Epoch 4/5,                      train loss: 0.5524, train metric: 73.09%, valid metric: 71.92%
Epoch 5/5,                      train loss: 0.5369, train metric: 74.17%, valid metric: 62.14%


In [65]:
torch.manual_seed(42)
bert_for_binary_clf = transformers.BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2, dtype=torch.float16
).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [66]:
encoding = bert_tokenizer(["This was a great movie!"])
with torch.no_grad():
    output = bert_for_binary_clf(
        input_ids = torch.tensor(encoding["input_ids"], device=device),
        attention_mask = torch.tensor(encoding["attention_mask"], device=device)
    )

In [67]:
output.logits

tensor([[0.2466, 0.3496]], device='cuda:0', dtype=torch.float16)

In [68]:
torch.softmax(output.logits, dim=-1)

tensor([[0.4744, 0.5259]], device='cuda:0', dtype=torch.float16)

In [69]:
with torch.no_grad():
    output = bert_for_binary_clf(
        input_ids = torch.tensor(encoding["input_ids"], device=device),
        attention_mask = torch.tensor(encoding["attention_mask"], device=device),
        labels= torch.tensor([1], device=device)
    )
output.loss


tensor(0.6431, device='cuda:0', dtype=torch.float16)

In [70]:
#Trainer API
def tokenize_batch(batch):
    return bert_tokenizer(batch["text"], truncation=True, max_length=200)

In [71]:
tok_imdb_train_set = imdb_train_set.map(tokenize_batch, batched=True)
tok_imdb_valid_set = imdb_valid_set.map(tokenize_batch, batched=True)
tok_imdb_test_set = imdb_test_set.map(tokenize_batch, batched=True)

In [72]:
def compute_accuracy(pred):
    return {"accuracy" : (pred.label_ids == pred.predictions.argmax(-1)).mean()}

In [73]:
from transformers import TrainingArguments


train_args = TrainingArguments(
    output_dir="my_imdb_model", num_train_epochs=2,
    per_device_train_batch_size=128, per_device_eval_batch_size=128,
    eval_strategy="epoch", logging_strategy='epoch', save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="accuracy",
    report_to='none'
)

In [74]:
from transformers import DataCollatorWithPadding, Trainer

trainer = Trainer(
    bert_for_binary_clf, train_args, train_dataset=tok_imdb_train_set, eval_dataset=tok_imdb_valid_set, compute_metrics=compute_accuracy,
    data_collator = DataCollatorWithPadding(bert_tokenizer)
)

train_output = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,4.560218,nan,0.498800
2,0.000000,nan,0.498800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

In [76]:
from transformers import pipeline

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
classifier_imdb = pipeline("sentiment-analysis", model=model_name, truncation=True, max_length=512)

classifier_imdb(train_reviews[:10])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9996108412742615},
 {'label': 'POSITIVE', 'score': 0.9998623132705688},
 {'label': 'NEGATIVE', 'score': 0.9943684935569763},
 {'label': 'POSITIVE', 'score': 0.997913658618927},
 {'label': 'POSITIVE', 'score': 0.999544084072113},
 {'label': 'NEGATIVE', 'score': 0.9845332503318787},
 {'label': 'POSITIVE', 'score': 0.9859278202056885},
 {'label': 'POSITIVE', 'score': 0.9993758797645569},
 {'label': 'POSITIVE', 'score': 0.9978922009468079},
 {'label': 'NEGATIVE', 'score': 0.9997020363807678}]

In [77]:
type(train_reviews)

list

In [79]:
classifier_imdb('i AM FROM THE USA')

[{'label': 'POSITIVE', 'score': 0.9642282128334045}]

In [81]:
classifier_imdb(["I am from Romania", 'I am from Republic of Moldova', 'I am from Iraq'])

[{'label': 'POSITIVE', 'score': 0.9961787462234497},
 {'label': 'POSITIVE', 'score': 0.9877511262893677},
 {'label': 'NEGATIVE', 'score': 0.9706069231033325}]

In [85]:
model_name = "huggingface/distilbert-base-uncased-finetuned-mnli"
classifier_mnli = pipeline("text-classification", model = model_name)
classifier_mnli([
    "She loves me. [SEP] She loves me not . [SEP]",
    "Alice just woke up. [SEP] Alice is awake. [SEP]",
    "I like dogs. [SEP] Everyone likes dogs. [SEP]"
])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'contradiction', 'score': 0.9717152714729309},
 {'label': 'entailment', 'score': 0.9119167327880859},
 {'label': 'neutral', 'score': 0.9509281516075134}]